# Printed-20 orientation inspection

Milestone 3: inspect a lightweight, replaceable number-ring matcher. No OCR, model downloads, segmentation, dart localization, or scoring. The scoring-board view is unchanged. Photos stay local.

A manually identified, unobscured `20` supplies the **board-style-specific template**. It is not a validation image. Other board fonts/sizes may require a new template/radial range. Match confidence is heuristic, not a probability. A wrong uncertain best guess is possible when the real label is obscured.

In [ ]:
import json
from functools import partial
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from dartcv import (
    Printed20OrientationStage, extract_printed20_template,
    format_timing_summary, normalize_board_view, prepare_board,
)
from dartcv.color_dart_inspection import plot_orientation

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
IMAGE_DIR = ROOT / 'data/images/multi_dart'
MANIFEST = json.loads((ROOT / 'data/annotations/color_dart_orientation.json').read_text())
PHOTO_PATHS = sorted(IMAGE_DIR.glob('*.jpg'))
if not PHOTO_PATHS:
    raise FileNotFoundError('Restore data/images using the README DVC workflow first.')
NORMALIZER = partial(normalize_board_view, number_ring_radius_mm=240.0)

def read_photo(name):
    image = cv2.imread(str(IMAGE_DIR / name))
    if image is None:
        raise FileNotFoundError(IMAGE_DIR / name)
    return image

def source_label_angle(preparation, source_xy):
    point = np.linalg.inv(preparation.board.model_to_source) @ np.array([*source_xy, 1.0])
    point = point[:2] / point[2]
    return float(np.rad2deg(np.arctan2(point[0], -point[1])))


## Prepare the explicit reference template once

The manifest stores approximate manually inspected source-image label centers, not angles predicted by the matcher. Projection into the detected board model converts that independent observation to an angle. No source imagery is embedded in this notebook. The annular crop spans radii 202–228 mm and 14 degrees; inspect it before trusting the matcher.

In [ ]:
reference = MANIFEST['reference']
reference_preparation = prepare_board(
    read_photo(reference['image']), board_normalization=NORMALIZER,
)
if reference_preparation.failure is not None:
    raise RuntimeError(reference_preparation.failure)
reference_angle = source_label_angle(reference_preparation, reference['label_center_source_px'])
template = extract_printed20_template(
    reference_preparation.board_view, label_angle_deg=reference_angle,
)
ORIENTATION_STAGE = Printed20OrientationStage(template)
plt.figure(figsize=(4, 3))
plt.imshow(cv2.cvtColor(template, cv2.COLOR_BGR2RGB))
plt.title('Reference 20 — inspect this template')
plt.axis('off')
plt.show()
print(format_timing_summary(reference_preparation))


## Select and inspect a different photograph

Change `IMAGE_INDEX`, then rerun this section. Index 1 has a visible label; index 0 has an obscured label and should **not** be trusted. The search covers all 360 degrees, including the wraparound seam, at four samples/degree. It tests template scales 0.9, 1.0 and 1.1. Missing source pixels cannot contribute matches.

A supported match below strength 0.65, or with less than 0.10 separation from the next distinct candidate, is uncertain. Strength below 0.25 fails explicitly. These provisional thresholds are not calibrated across board styles.

In [ ]:
IMAGE_INDEX = 1
image_path = PHOTO_PATHS[IMAGE_INDEX]
preparation = prepare_board(read_photo(image_path.name), board_normalization=NORMALIZER)
if preparation.failure is not None:
    raise RuntimeError(preparation.failure)
orientation = ORIENTATION_STAGE(preparation.board_view, diagnostics=True)
print(image_path.name)
print(format_timing_summary(preparation))
for step, milliseconds in orientation.substep_timings_ms.items():
    print(f'{step}: {milliseconds:.2f} ms')
print('Orientation:', orientation.value if orientation.value is not None else orientation.failure)
print('Candidates [clockwise label angle, correlation]:')
print(orientation.artifacts['orientation_candidates'])
figure = plot_orientation(orientation)
display(figure)


## Check observed source coverage and an independent annotation

Black padding is not board evidence. This mask records source coverage, unlike the nominal scoring-board disk. `correction_deg` is a **clockwise image-space rotation in the rectified model**, i.e. the negative of the matched label angle; it is not an OpenCV `getRotationMatrix2D` angle (which is counterclockwise). No scoring transform is silently modified.

In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(preparation.board_view.number_ring.source_coverage, cmap='gray', vmin=0, vmax=255)
plt.title('Wider-view source coverage (white = observed)')
plt.axis('off')
plt.show()
annotation = next((entry for entry in MANIFEST['checks'] if entry['image'] == image_path.name), None)
if annotation is not None:
    print('Manually inspected annotation:', annotation)
    if annotation['label_center_source_px'] is not None and orientation.value is not None:
        annotated_angle = source_label_angle(preparation, annotation['label_center_source_px'])
        error = (orientation.value.correction_deg + annotated_angle + 180.0) % 360.0 - 180.0
        print(f'Correction error against approximate label center: {error:.2f} degrees')
else:
    print('This photograph has no manual orientation annotation yet.')


## Integration and next milestone

Inject `orientation=ORIENTATION_STAGE` and `board_normalization=NORMALIZER` into `ColorBasedDartRecognitionPipeline` once real downstream stages exist. The existing `OrientationStage` signature is unchanged. `diagnostics=False` avoids diagnostic capture; the normalizer's wider view remains opt-in. Stage timings exclude rendering, image I/O and template-reference preparation.

Next: geometry-derived field-interior masks and robust same-color models. Do not mistake number matching for dart detection or scoring. See `docs/architecture.md` for the durable handoff.